# From the LHC to the HL-LHC: top-quark pairs in Run 2 and at 200 pile-up collisions

The **High-Luminosity LHC (HL-LHC)** will collect about 3000 fb$^{-1}$ of proton–proton
collisions, roughly twenty times the 140 fb$^{-1}$ of Run 2. The price is **pile-up**: about 200
simultaneous collisions in every bunch crossing. ATLAS is being upgraded for this with an
all-silicon **Inner Tracker (ITk)** that tracks charged particles out to $|\eta| = 4$ (2.5 today), a
**High-Granularity Timing Detector (HGTD)** in the forward region, and new trigger and readout
electronics.

This notebook compares top-quark pair ($t\bar{t}$) events simulated for Run 2 and for the HL-LHC,
in four steps:

1. **Pile-up**: how many collisions, and how many vertices we can see.
2. **What pile-up does**: extra jets, and a blurrier missing transverse momentum.
3. **Wider detector coverage**: what lies beyond $|\eta| = 2.5$.
4. **Luminosity**: how much better a simple measurement gets with 3000 fb$^{-1}$.

### The two samples

| | Run 2 | HL-LHC |
|---|---|---|
| Dataset | `mc_410470.PhPy8EG_A14_ttbar_hdamp258p75_nonallhad` (ATLAS Open Data) | `mc23_valid.601229.PhPy8EG_A14_ttbar_hdamp258p75_SingleLep` (e8514_e8586_s4680_r17821) |
| $\sqrt{s}$ | 13 TeV | 13.6 TeV |
| Pile-up | 2015–16 conditions; $\mu$ not stored in the ntuple | $\langle\mu\rangle \approx 200$ |
| $t\bar{t}$ decays | at least one leptonic W (single lepton **and** dilepton) | exactly one leptonic W (single lepton only) |
| Generator | Powheg + Pythia 8, A14 tune, $h_\mathrm{damp} = 258.75$ GeV | same |

> **Caveats — this is not an official ATLAS performance result.**
> * The HL-LHC sample is a *validation* sample (`mc23_valid`, s4680/r17821) processed with
>   preliminary Phase-II software. Objects keep their derivation calibration
>   (`recalibratePhyslite: false`); the jet energy scale is measured below to be about 9% low.
> * No scale factors, no trigger, and Run 3 identification, isolation and pile-up-jet
>   working points that are not tuned for $\mu = 200$. b-tagging (GN2HLv01) has no calibration;
>   its 77% working point is defined from truth labels in this sample.
> * The Run 2 Open Data ntuple only keeps jets with $|\eta| < 2.5$, and reconstructs only about
>   73% of its own truth jets above 60 GeV, so it has fewer jets than expected.
> * Only 10,000 events of each sample are used.

## First-time setup

This cell installs the required Python packages. It only needs to be run the first time you open
this notebook on your computer.

In [ ]:
# install required packages
import sys
!pip install atlasopenmagic
from atlasopenmagic import install_from_environment
install_from_environment()

## Setup

All the reading, selections and plotting live in `hllhc_ttbar_compare.py`, next to this notebook. It
converts both ntuples to the same layout (GeV, sorted by $p_T$), so every selection below is
applied to both samples in exactly the same way.

In [ ]:
import awkward as ak
import matplotlib.pyplot as plt
import numpy as np

import hllhc_ttbar_compare as hc

RUN2, HLLHC = hc.STYLES['run2'], hc.STYLES['hllhc']
N_EVENTS = 10_000  # Run 2 events to stream from the Open Data portal
# TODO: the HL-LHC ntuple is not public yet; replace with its Open Data URL once it is
HLLHC_FILE = hc.HLLHC_FILE

run2 = hc.load_run2(n_events=N_EVENTS)
hllhc = hc.load_hllhc(HLLHC_FILE)
fields = hc.check_schema(run2, hllhc)
print(f'Run 2: {len(run2)} events, HL-LHC: {len(hllhc)} events, {len(fields)} common fields')

Because the two samples contain different $t\bar{t}$ decays, we first keep only events with
**exactly one truth electron or muon** (and no tau) at generator level, before any detector
effects. After this filter the two samples agree at truth level: lepton $p_T$, jet $p_T$ and
missing transverse momentum match within statistics (see `validation/VALIDATION.md`). Any
difference in the plots of Sections 2 and 3 then comes from the detector, pile-up or the ntuple
production, not from the physics process.

In [ ]:
run2_1l = run2[hc.truth_single_lepton(run2)]
hllhc_1l = hllhc[hc.truth_single_lepton(hllhc)]
print(f'Truth single-lepton events: Run 2 {len(run2_1l)}, HL-LHC {len(hllhc_1l)}')

## 1. Pile-up

$\mu$ is the average number of proton–proton collisions per bunch crossing. Only one of them is the
$t\bar{t}$ event; the rest are mostly soft collisions that spray low-energy particles through the
detector. The Run 2 Open Data ntuple doesn't store $\mu$, so we mark the published Run 2 average.
This HL-LHC test sample was simulated with integer $\mu$ values between 190 and 210; the uneven
heights come from how events were spread over those values in the production, not from physics.

In [ ]:
RUN2_AVERAGE_MU = 33.7  # ATLAS luminosity public results, 2015-2018 average (not from this ntuple)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(hc.to_numpy(hllhc.mu), bins=np.arange(-0.5, 221), color=HLLHC['color'])
ax.text(186, ax.get_ylim()[1] * 0.92, 'HL-LHC sample\n' + r'$\mu$ = 190–210', ha='right', va='top',
        fontsize=10)
hc.vline(ax, RUN2_AVERAGE_MU, rf'Run 2 average $\langle\mu\rangle$ = {RUN2_AVERAGE_MU}' '\n(published)',
         color=RUN2['color'], y=0.92)
ax.set(xlim=(0, 225), xlabel=r'Average interactions per bunch crossing, $\mu$', ylabel='Events',
       title='Collisions per bunch crossing')
plt.show()

Each collision should leave a **primary vertex**: a point on the beam line where several tracks
meet. At the HL-LHC we reconstruct only about half as many vertices as there are collisions: many
collisions produce too few tracks, and others lie so close to a neighbour along the beam that the
two are merged. This is why the HGTD adds a time measurement, to separate collisions that overlap
in space.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(hc.to_numpy(hllhc.npv), bins=np.arange(50, 151, 2), color=HLLHC['color'])
hc.vline(ax, np.mean(hllhc.npv), f'mean {np.mean(hllhc.npv):.0f} vertices', color=hc.INK)
hc.vline(ax, np.mean(hllhc.mu), rf'$\langle\mu\rangle$ = {np.mean(hllhc.mu):.0f} collisions',
         color=hc.INK, ha='right', linestyle=':')
ax.set(xlim=(50, 215), xlabel='Number of reconstructed primary vertices', ylabel='Events',
       title='HL-LHC: reconstructed vertices per event')
plt.show()
print(f'HL-LHC: mean mu = {np.mean(hllhc.mu):.1f}, mean vertices = {np.mean(hllhc.npv):.1f}, '
      f'ratio = {np.mean(hllhc.npv) / np.mean(hllhc.mu):.2f}')

## 2. What pile-up does

### 2.1 Extra jets, and removing them

Pile-up collisions produce extra, mostly soft jets. ATLAS rejects them with the **Jet Vertex
Tagger** (here the neural-network version, NNJvt), which checks with tracks whether a jet comes
from the $t\bar{t}$ vertex. It is applied to jets with $p_T < 60$ GeV and $|\eta| < 2.4$, where
tracks exist. The plot shows **jets per event**, so you can see how many jets JVT removes.

In [ ]:
def per_event(values, events):
    """Weights that turn a histogram of objects into objects per event."""
    return np.full(len(hc.to_numpy(values)), 1 / len(events))


hl_all = hllhc_1l.jet
hl_clean = hl_all[hl_all.jvt & hl_all.passes_or]
r2_clean = run2_1l.jet[run2_1l.jet.jvt]
samples = [(r2_clean.eta, RUN2, per_event(r2_clean.eta, run2_1l)),
           (hl_clean.eta, HLLHC, per_event(hl_clean.eta, hllhc_1l)),
           (hl_all.eta, hc.style('hllhc', 'HL-LHC before JVT', linestyle=':', marker='x'),
            per_event(hl_all.eta, hllhc_1l))]
fig = hc.compare(samples, np.linspace(-4.5, 4.5, 37), r'Jet $\eta$', density=False,
                 ylabel='Jets per event',
                 title=r'Jets with $p_T > 20$ GeV (the Run 2 ntuple keeps $|\eta| < 2.5$ only)')
for sign in (-1, 1):
    hc.shade(fig.axes[0], sign * 2.4, sign * 4.5, 'no tracks:\nJVT not applied', y=0.55)

Inside $|\eta| < 2.5$, JVT removes most of the extra HL-LHC jets. Beyond $|\eta| = 2.4$ there are no
tracks today, so the forward jets stay; Section 3 shows that most of them are pile-up.

A good way to see pile-up is to count jets above a $p_T$ threshold. Pile-up jets are soft, so they
matter most at low thresholds. **Truth jets** (built from generated particles, without the
detector or pile-up) show what the $t\bar{t}$ event really contains.

In [ ]:
THRESHOLDS = np.arange(20, 101, 10)  # GeV; try other values


def jets_per_event(jets, events):
    central = jets[abs(jets.eta) < 2.5]
    return [ak.sum(central.pt > t) / len(events) for t in THRESHOLDS]


curves = [  # colour = sample, line style = what is counted
    (hl_all, hllhc_1l, 'HL-LHC, before JVT', HLLHC, (0, (1, 1.5)), 'x'),
    (hl_clean, hllhc_1l, 'HL-LHC, with JVT', HLLHC, '-', 's'),
    (hllhc_1l.truth_jet, hllhc_1l, 'HL-LHC truth jets', HLLHC, '--', 'none'),
    (run2_1l.jet[run2_1l.jet.jvt], run2_1l, 'Run 2, with JVT', RUN2, '-', 'o'),
    (run2_1l.truth_jet, run2_1l, 'Run 2 truth jets', RUN2, '--', 'none'),
]
fig, ax = plt.subplots(figsize=(8, 5))
for jets, events, label, st, linestyle, marker in curves:
    ax.plot(THRESHOLDS, jets_per_event(jets, events), color=st['color'], linestyle=linestyle,
            marker=marker, markersize=5, label=label)
ax.set(xlabel=r'Jet $p_T$ threshold [GeV]', ylabel=r'Central jets ($|\eta|<2.5$) per event',
       ylim=(0, None), xlim=(THRESHOLDS[0] - 3, THRESHOLDS[-1] + 3),
       title='Jets above a threshold, truth single-lepton events')
ax.legend(loc='upper right')
plt.show()

* Before JVT the HL-LHC sample has clearly more jets than truth near 20 GeV: pile-up jets.
  After JVT the excess is gone.
* At higher thresholds HL-LHC jets fall about 15% below their truth jets. That is the jet energy
  scale of this validation sample: reconstructed jets carry about 91% of the truth jet energy, so
  fewer pass each threshold. A calibration would fix this.
* The Run 2 Open Data ntuple falls even further below *its* truth jets at every threshold. This
  comes from how that ntuple was produced (it misses about a quarter of the jets), not from the
  Run 2 detector.

### 2.2 A blurrier missing transverse momentum

Neutrinos escape the detector; we infer them from the momentum imbalance $E_T^{\mathrm{miss}}$ in
the transverse plane. Comparing each event's reconstructed $E_x^{\mathrm{miss}}$ and
$E_y^{\mathrm{miss}}$ with the truth value (the real neutrino momentum) measures the **resolution**:
the narrower the distribution, the better.

In [ ]:
def met_residuals(events):
    return np.concatenate([hc.to_numpy(events.met_mpx - events.truth_met_mpx),
                           hc.to_numpy(events.met_mpy - events.truth_met_mpy)])


res_run2, res_hl = met_residuals(run2_1l), met_residuals(hllhc_1l)
fig = hc.compare([(res_run2, RUN2), (res_hl, HLLHC)], np.linspace(-100, 100, 41),
                 r'$E_{x,y}^{\mathrm{miss}}$ (reco) $-$ $E_{x,y}^{\mathrm{miss}}$ (truth) [GeV]',
                 title='Missing transverse momentum resolution')
for name, r in (('Run 2', res_run2), ('HL-LHC', res_hl)):
    print(f'{name}: RMS = {np.std(r):.1f} GeV')

Within the HL-LHC sample, the resolution gets worse as the number of vertices grows: every extra
collision adds random energy.

In [ ]:
npv = hc.to_numpy(hllhc_1l.npv)
dx = hc.to_numpy(hllhc_1l.met_mpx - hllhc_1l.truth_met_mpx)
edges = np.quantile(npv, np.linspace(0, 1, 6))
centers, rms, err = [], [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    sel = (npv >= lo) & (npv <= hi)
    centers.append(npv[sel].mean())
    rms.append(dx[sel].std())
    err.append(dx[sel].std() / np.sqrt(2 * sel.sum()))
run2_rms = np.std(hc.to_numpy(run2_1l.met_mpx - run2_1l.truth_met_mpx))
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.errorbar(centers, rms, yerr=err, fmt='s-', color=HLLHC['color'], markersize=7, capsize=0)
ax.text(centers[-1], rms[-1] + 2.5, 'HL-LHC', ha='center', va='bottom', fontsize=10)
ax.axhline(run2_rms, color=RUN2['color'], linestyle='--', linewidth=1.5)
ax.text(centers[0], run2_rms - 1.5, 'Run 2, all events (number of vertices not stored)',
        va='top', fontsize=10)
ax.set(xlabel='Number of reconstructed primary vertices (bins of equal size)',
       ylabel=r'RMS of $E_x^{\mathrm{miss}}$ (reco $-$ truth) [GeV]', ylim=(0, 1.2 * max(rms)),
       title='Missing transverse momentum resolution vs. pile-up')
plt.show()

## 3. Wider detector coverage

### 3.1 What lies beyond $|\eta| = 2.5$?

The generator produces leptons at all angles. Both samples contain the same fraction of truth
leptons beyond $|\eta| = 2.5$, the edge of today's tracker. The ITk extends tracking to
$|\eta| = 4$, which could recover some of them. (In this HL-LHC ntuple the reconstructed electrons
and muons still stop at $|\eta| = 2.5$, so this gain cannot yet be shown with reconstructed
objects.)

In [ ]:
def truth_lepton_eta(events):
    return events.truth_lep.eta[events.truth_lep.pt > 25]


fig = hc.compare([(truth_lepton_eta(run2), RUN2), (truth_lepton_eta(hllhc), HLLHC)],
                 np.linspace(-4.5, 4.5, 37), r'Truth lepton $\eta$ ($p_T > 25$ GeV)',
                 title='Truth electrons and muons, all events')
ax = fig.axes[0]
hc.shade(ax, -2.5, 2.5, 'Run 2 tracker')
for sign in (-1, 1):
    hc.shade(ax, sign * 2.5, sign * 4.0, 'ITk\nextension', color='#fbe3d8', y=0.45)
for name, events in (('Run 2', run2), ('HL-LHC', hllhc)):
    eta = abs(hc.to_numpy(truth_lepton_eta(events)))
    print(f'{name}: {np.mean((eta > 2.5) & (eta < 4)):.1%} of truth leptons have 2.5 < |eta| < 4')

### 3.2 Forward jets: mostly pile-up

The calorimeters already reach $|\eta| = 4.9$ (the Run 2 Open Data ntuple just keeps $|\eta| < 2.5$).
Using the simulation's truth labels we can split the HL-LHC jets into jets from the $t\bar{t}$
collision ("hard scatter") and pile-up jets. Beyond $|\eta| \approx 2.5$, where JVT cannot be used,
pile-up jets take over; beyond $|\eta| = 4$ almost every jet is pile-up. Track-based rejection with
the ITk, and timing from the HGTD, are aimed at exactly this region.

In [ ]:
FORWARD_PT_MIN = 20  # GeV; try 30 or 50 to see how the pile-up fraction changes
jets = hl_clean[hl_clean.pt > FORWARD_PT_MIN]
edges = np.linspace(-4.5, 4.5, 37)
hard, _ = np.histogram(hc.to_numpy(jets.eta[jets.hs == 1]), edges)
pileup, _ = np.histogram(hc.to_numpy(jets.eta[jets.hs == 0]), edges)
fig, ax = plt.subplots(figsize=(8, 5))
n = len(hllhc_1l)
ax.stairs(hard / n, edges, fill=True, color=HLLHC['color'], edgecolor='white', linewidth=0.6,
          label='hard-scatter jets (from the $t\\bar{t}$ collision)')
ax.stairs((hard + pileup) / n, edges, baseline=hard / n, fill=True, color='#b9b9b9',
          edgecolor='white', linewidth=0.6, label='pile-up jets')
for sign in (-1, 1):
    hc.vline(ax, sign * 4.0, 'ITk edge' if sign > 0 else None, ha='right')
ax.set(xlim=(-4.5, 4.5), ylim=(0, 1.25 * ax.get_ylim()[1]), xlabel=r'Jet $\eta$',
       ylabel='Jets per event (after JVT)',
       title=rf'HL-LHC jets with $p_T > {FORWARD_PT_MIN}$ GeV, split with truth labels')
ax.legend(loc='upper center')
plt.show()
for lo, hi in ((0, 2.5), (2.5, 4.0), (4.0, 4.5)):
    region = (abs(jets.eta) >= lo) & (abs(jets.eta) < hi)
    print(f'{lo} < |eta| < {hi}: pile-up fraction {1 - ak.mean(jets.hs[region] == 1):.0%}')

## 4. Luminosity: 140 fb$^{-1}$ versus 3000 fb$^{-1}$

We now select $t\bar{t}$ events as an analysis would, **with the same cuts on both samples**:
exactly one isolated electron or muon ($p_T > 25$ GeV, $|\eta| < 2.5$), at least four jets
($p_T > 30$ GeV, $|\eta| < 2.5$, passing JVT) and at least one b-tagged jet. The two highest-$p_T$
jets that are *not* b-tagged are taken as the decay products of the hadronic W boson, and their
invariant mass $m_{jj}$ shows a peak near 80 GeV.

Expected event numbers are $N = \sigma \times \varepsilon \times L$. The Run 2 cross section comes
from the Open Data file. **The cross section of the HL-LHC sample is not yet known here (TODO)**,
so for 3000 fb$^{-1}$ we assume the same selected cross section as Run 2: only the luminosity and
the shape of the distribution change.

In [ ]:
LUMI_RUN2, LUMI_HLLHC = 140, 3000  # fb^-1; try other values


def selected_mjj(events):
    """m_jj of selected events and the indices of those events in `events`."""
    mask = ak.to_numpy(hc.cutflow(events)[-1][1])
    m_jj, _ = hc.hadronic_top(hc.baseline_jets(events[mask]))
    has_w = ~ak.to_numpy(ak.is_none(m_jj))
    return ak.to_numpy(ak.fill_none(m_jj, 0))[has_w], np.flatnonzero(mask)[has_w]


mjj_run2, idx_run2 = selected_mjj(run2)
mjj_hl, idx_hl = selected_mjj(hllhc)


def weights(sample, lumi):
    """Run 2: sigma x L from the file. HL-LHC: same selected yield as Run 2 (assumption)."""
    run2_yield = hc.event_weights(run2, lumi)[idx_run2].sum()
    if sample == 'run2':
        return hc.event_weights(run2, lumi)[idx_run2]
    w = hc.event_weights(hllhc, lumi)
    if w is not None:
        return w[idx_hl]
    w = ak.to_numpy(hllhc.mc_weight)[idx_hl].astype(float)
    return w / w.sum() * run2_yield


print(f'Selected events (MC): Run 2 {len(mjj_run2)}, HL-LHC {len(mjj_hl)}')
print(f'Expected Run 2 events at {LUMI_RUN2} fb^-1: {weights("run2", LUMI_RUN2).sum():,.0f}')

First the **shape**: the HL-LHC peak is wider, because pile-up adds energy to the jets and the jets
of this validation sample are not yet calibrated.

In [ ]:
fig = hc.compare([(mjj_run2, RUN2), (mjj_hl, HLLHC)], np.linspace(40, 130, 31), r'$m_{jj}$ [GeV]',
                 title='Hadronic W candidate (two leading untagged jets)')
for name, values in (('Run 2', mjj_run2), ('HL-LHC', mjj_hl)):
    fit = hc.fit_peak(values)
    print(f"{name}: peak {fit['mean']:.1f} GeV, width {fit['sigma']:.1f} +- {fit['sigma_err']:.1f} GeV")

How precisely can we measure the position of the peak? We fit a Gaussian on a straight-line
background to the *expected* distribution, using the statistical uncertainty of each bin at the
chosen luminosity. Swapping shapes and luminosities separates the two effects: more data
shrinks the uncertainty by $\sqrt{L}$, while the wider HL-LHC peak (pile-up, uncalibrated jets)
gives some of that back.

Notice how small the numbers are: Run 2 already provides millions of $t\bar{t}$ events, so the
statistical uncertainty on the peak is tiny compared with systematic effects such as the jet
energy scale (in this sample alone the HL-LHC peak sits about 1.5 GeV low). Top-quark pairs are
so common that their measurements are limited by systematic uncertainties already in Run 2. The
HL-LHC's 3000 fb$^{-1}$ matter most for rare processes, where Run 2 collected only a handful of
events.

In [ ]:
print(f"{'shape':>8} {'L [fb^-1]':>10} {'events':>9} {'peak [GeV]':>11} {'width [GeV]':>12} "
      f"{'stat. unc. on peak':>19}")
for shape, values in (('Run 2', mjj_run2), ('HL-LHC', mjj_hl)):
    for lumi in (LUMI_RUN2, LUMI_HLLHC):
        w = weights('run2' if shape == 'Run 2' else 'hllhc', lumi)
        fit = hc.fit_peak(values, weights=w)
        print(f"{shape:>8} {lumi:>10} {w.sum():>9,.0f} {fit['mean']:>11.1f} {fit['sigma']:>12.1f} "
              f"{fit['mean_err']:>15.3f} GeV")

In [ ]:
lumis = np.geomspace(10, 10_000, 30)
fig, ax = plt.subplots(figsize=(8, 4.8))
for shape, values, sample in (('Run 2', mjj_run2, 'run2'), ('HL-LHC', mjj_hl, 'hllhc')):
    reference = hc.fit_peak(values, weights=weights(sample, LUMI_RUN2))['mean_err']
    st = hc.STYLES[sample]
    # Asimov fits scale exactly as 1/sqrt(L), so one fit is enough
    curve = reference * np.sqrt(LUMI_RUN2 / lumis)
    ax.plot(lumis, curve, color=st['color'], linestyle=st['linestyle'])
    ax.text(lumis[2], curve[2], f'  {shape} peak shape', va='bottom', fontsize=10)
for lumi, text in ((LUMI_RUN2, 'Run 2'), (LUMI_HLLHC, 'HL-LHC')):
    hc.vline(ax, lumi, f'{text}, {lumi} fb$^{{-1}}$', linestyle=':')
ax.set(xscale='log', xlim=(10, 10_000), xlabel=r'Integrated luminosity [fb$^{-1}$]',
       ylabel='Statistical uncertainty on the peak [GeV]', ylim=(0, None),
       title=r'How precisely the $m_{jj}$ peak position is known from statistics alone')
plt.show()

## Summary

| | What we saw | Why |
|---|---|---|
| Pile-up | $\langle\mu\rangle \approx 200$, about 100 reconstructed vertices per event | Higher luminosity; vertices merge or have too few tracks |
| Jets | Many extra soft jets, mostly removed by JVT in the central region | Pile-up |
| Forward jets | Beyond $|\eta| = 4$ almost all jets are pile-up | No tracks yet to reject them; the ITk and HGTD target this |
| $E_T^{\mathrm{miss}}$ | Wider resolution, growing with the number of vertices | Random energy from pile-up |
| Leptons | Same truth fraction beyond $|\eta| = 2.5$ in both samples | Potential ITk gain; not yet in the reconstructed ntuple |
| Luminosity | Statistical precision on the W peak improves by $\sqrt{3000/140} \approx 4.6$, partly offset by the broader peak; for inclusive $t\bar{t}$ it is already far below systematic uncertainties | More data, broader peak |

### Things to try

1. Change `THRESHOLDS` in Section 2.1: where does the HL-LHC jet count drop below truth?
2. Change `FORWARD_PT_MIN` in Section 3.2: does a higher threshold make forward jets usable?
3. Change `LUMI_HLLHC` in Section 4 and check that the uncertainty scales as $1/\sqrt{L}$.